# Task 2: Neural ODE for Mean SIR Dynamics

Train a parameter-conditioned Neural ODE to predict mean epidemic trajectories from the stochastic ensemble.

In [ ]:
!git clone https://github.com/invi-bhagyesh/sira.git 2>/dev/null || echo "already cloned"
%cd sira
!pip install -q -r requirements.txt

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.simulation.dataset import load_dataset, train_val_test_split
from src.models.neural_ode import NeuralODE, train_neural_ode

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
data = load_dataset('data/sir_dataset.npz')
train, val, test = train_val_test_split(data)
print(f'Train: {len(train["params"])}, Val: {len(val["params"])}, Test: {len(test["params"])}')

## 1. Train Neural ODE

MLP with 3 hidden layers (64 units, SiLU). Input: [s, i, r, beta, gamma]. Output: [ds/dt, di/dt, dr/dt].

In [ ]:
Path('checkpoints').mkdir(exist_ok=True)

model = NeuralODE(hidden_dim=64, n_layers=3, conservation_lambda=0.01)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params}')

CKPT = 'checkpoints/neural_ode.pt'
if os.path.exists(CKPT):
    print(f'Loading trained model from {CKPT}')
    model.load_state_dict(torch.load(CKPT, map_location=device))
    model.to(device)
    history = {'train_loss': [], 'val_loss': []}
else:
    history = train_neural_ode(
        model, train, val,
        epochs=500, lr=1e-3, batch_size=32,
        patience=50, noise_sigma=0.005, device=device
    )
    torch.save(model.state_dict(), CKPT)
    print(f'Saved to {CKPT}')

In [ ]:
if history['train_loss']:
    plt.figure(figsize=(8, 4))
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Neural ODE Training')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

## 2. Evaluate on Test Set

Compare predicted vs actual mean trajectories for held-out parameter combinations.

In [ ]:
model.eval()
t_grid = torch.tensor(test['t_grid'], dtype=torch.float32, device=device)
test_s = torch.tensor(test['mean_s'], dtype=torch.float32, device=device)
test_i = torch.tensor(test['mean_i'], dtype=torch.float32, device=device)
test_r = torch.tensor(test['mean_r'], dtype=torch.float32, device=device)
test_params = torch.tensor(test['params'][:, :2], dtype=torch.float32, device=device)
test_targets = torch.stack([test_s, test_i, test_r], dim=-1)

with torch.no_grad():
    x0 = test_targets[:, 0, :]
    pred = model(x0, test_params, t_grid).permute(1, 0, 2)  # (N, T, 3)

# metrics
mae = (pred - test_targets).abs().mean().item()
rel_err = ((pred - test_targets).abs() / (test_targets.abs() + 1e-8)).mean().item()

# peak infection error
pred_peak = pred[:, :, 1].max(dim=1).values
true_peak = test_targets[:, :, 1].max(dim=1).values
peak_rel_err = ((pred_peak - true_peak).abs() / (true_peak + 1e-8)).mean().item()

print(f'MAE: {mae:.6f}')
print(f'Mean relative error: {rel_err:.4f} ({rel_err*100:.2f}%)')
print(f'Peak infection relative error: {peak_rel_err:.4f} ({peak_rel_err*100:.2f}%)')

In [ ]:
# visualize 6 random test trajectories
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
t_np = t_grid.cpu().numpy()
pred_np = pred.cpu().numpy()
true_np = test_targets.cpu().numpy()

rng = np.random.default_rng(42)
indices = rng.choice(len(test['params']), size=6, replace=False)

for ax, idx in zip(axes.flat, indices):
    beta, gamma = test['params'][idx, 0], test['params'][idx, 1]
    for comp, color, label in zip(range(3), ['blue', 'red', 'green'], ['S', 'I', 'R']):
        ax.plot(t_np, true_np[idx, :, comp], '--', color=color, alpha=0.7, label=f'{label} (true)')
        ax.plot(t_np, pred_np[idx, :, comp], '-', color=color, lw=1.5, label=f'{label} (pred)')
    ax.set_title(f'beta={beta:.2f}, gamma={gamma:.2f}, R0={beta/gamma:.1f}', fontsize=9)
    ax.set_xlabel('Time')
    ax.grid(True, alpha=0.3)
    if ax == axes[0, 0]:
        ax.legend(fontsize=7, ncol=2)

plt.suptitle('Neural ODE: Predicted vs Actual Mean Trajectories (Test Set)', fontsize=12)
plt.tight_layout()
plt.show()